# SpeechLM API in vLLM example

In [ ]:
# example of llm streaming usage

from operator import ne
from vllm import SamplingParams
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.v1.engine.async_llm import AsyncLLM

import torch

model_path = "/home/vklimkov/.cache/huggingface/hub/models--TinyLlama--TinyLlama-1.1B-Chat-v1.0/snapshots/fe8a4ea1ffedaf415f4da2f062534de366a451e6/"
engine_args = AsyncEngineArgs(
    model=model_path,
    max_model_len=256,
    gpu_memory_utilization=0.8,
    #enforce_eager=True,
)
engine = AsyncLLM.from_engine_args(engine_args)
sampling_params = SamplingParams(max_tokens=20, skip_sampling=True)

tokens = engine.tokenizer.encode("My name is")
inputs = {
    "prompt_token_ids": [0] * len(tokens),
    "custom_inputs": {
        "custom_in_tokens": torch.tensor(tokens, dtype=torch.int32),
    }
}

sampled_tokens = []
async for output in engine.generate(
    inputs,
    sampling_params=sampling_params,
    request_id="1",
):
    new_token = output.outputs[0].custom_outputs["custom_out_tokens"][-1].item()
    sampled_tokens.append(new_token)
    if not output.finished:
        await engine.append_request(
            request_id="1",
            custom_inputs={"custom_in_tokens": torch.tensor([new_token], dtype=torch.int32)}
        )

print(f">>> sampledtokens: [{str(sampled_tokens)}]", flush=True)
text = engine.tokenizer.decode(sampled_tokens)
print(f">>> sampledtext: [{text}]", flush=True)

In [1]:
# Same example, but with shared-memory decode channel enabled.
#
# When shm_decode=True, add_request() automatically creates a SHM
# channel.  Each decode iteration uses a single decode_step() call
# that writes inputs to shared memory, signals the core, and waits
# for the output — all in one await.
#
# Requirements:
#   - Model config.json must define custom_input_specs AND
#     custom_output_specs.

from vllm import SamplingParams
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.v1.engine.async_llm import AsyncLLM
import torch

engine_args = AsyncEngineArgs(
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_model_len=256,
    gpu_memory_utilization=0.8,
    shm_decode=True,  # enable shared-memory decode channel
    input_coalesce_timeout_ms=5,
)
engine = AsyncLLM.from_engine_args(engine_args)
sampling_params = SamplingParams(max_tokens=100, skip_sampling=True)

tokens = engine.tokenizer.encode("My name is")
inputs = {
    "prompt_token_ids": [0] * len(tokens),
    "custom_inputs": {
        "custom_in_tokens": torch.tensor(tokens, dtype=torch.int32),
    },
}

request_id = "shm_1"

# 1. Submit the request.  shm_decode=True means add_request()
#    internally creates and registers a SHM channel.
queue = await engine.add_request(request_id, inputs, sampling_params)

# 2. Prefill output arrives via the normal ZMQ/output-processor path.
prefill_output = await queue.get()
last_token = prefill_output.outputs[0].custom_outputs["custom_out_tokens"][-1].item()


sampled_tokens = [last_token]

# 3. Decode loop — single decode_step() call per iteration.
#    Writes inputs to SHM, signals the core, blocks in futex until
#    output is ready, and returns.  Synchronous — no asyncio bounce.
for _ in range(sampling_params.max_tokens - 1):
    outputs = engine.decode_step_shm(
        request_id,
        custom_inputs={
            "custom_in_tokens": torch.tensor([last_token], dtype=torch.int32),
        },
    )
    last_token = int(outputs["custom_out_tokens"][0])
    sampled_tokens.append(last_token)

# 4. Clean up (also closes the SHM channel).
await engine.abort(request_id)


print(f">>> sampledtokens: [{str(sampled_tokens)}]", flush=True)
text = engine.tokenizer.decode(sampled_tokens)
print(f">>> sampledtext: [{text}]", flush=True)

/home/vklimkov/miniconda3/envs/vllm/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/vklimkov/miniconda3/envs/vllm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 03-03 14:30:17 [__init__.py:224] Automatically detected platform cuda.
INFO 03-03 14:30:18 [model.py:661] Resolved architecture: LlamaForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 03-03 14:30:18 [model.py:1760] Using max model len 256


2026-03-03 14:30:18,828	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


...

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  4.90it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  4.85it/s]
(EngineCore_DP0 pid=761829) 


(EngineCore_DP0 pid=761829) INFO 03-03 14:30:22 [default_loader.py:309] Loading weights took 0.22 seconds
(EngineCore_DP0 pid=761829) INFO 03-03 14:30:22 [gpu_model_runner.py:3052] Model loading took 2.0513 GiB and 1.001405 seconds
(EngineCore_DP0 pid=761829) INFO 03-03 14:30:23 [backends.py:594] Using cache directory: /home/vklimkov/.cache/vllm/torch_compile_cache/79388f67a8/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=761829) INFO 03-03 14:30:23 [backends.py:608] Dynamo bytecode transform time: 1.34 s
(EngineCore_DP0 pid=761829) INFO 03-03 14:30:24 [backends.py:179] Directly load the compiled graph(s) for dynamic shape from the cache, took 0.484 s
(EngineCore_DP0 pid=761829) INFO 03-03 14:30:24 [monitor.py:33] torch.compile takes 1.34 s in total
(EngineCore_DP0 pid=761829) INFO 03-03 14:30:25 [gpu_worker.py:314] Available KV cache memory: 35.63 GiB
(EngineCore_DP0 pid=761829) INFO 03-03 14:30:25 [kv_cache_utils.py:1133] GPU KV cache size: 1,698,016 tokens
(EngineCor

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 35/35 [00:00<00:00, 65.18it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 19/19 [00:00<00:00, 77.91it/s]


(EngineCore_DP0 pid=761829) INFO 03-03 14:30:26 [gpu_model_runner.py:3943] Graph capturing finished in 1 secs, took 0.30 GiB
(EngineCore_DP0 pid=761829) INFO 03-03 14:30:26 [core.py:245] init engine (profile, create kv cache, warmup model) took 4.13 seconds
INFO 03-03 14:30:26 [loggers.py:148] Engine 000: vllm cache_config_info with initialization after num_gpu_blocks is: 106126
WARNING 03-03 14:30:26 [async_llm.py:289] Processor has been moved under OpenAIServing and will be removed from AsyncLLM in v0.13.
(EngineCore_DP0 pid=761829) INFO 03-03 14:30:26 [gc_utils.py:40] GC Debug Config. enabled:False,top_objects:-1
INFO 03-03 14:30:27 [async_llm.py:448] decode_step_shm[shm_1]: write+signal=0.092ms, wait_output=4.962ms (core=4.768ms, asyncio_bounce=0.193ms), consume=0.008ms, total=5.061ms
INFO 03-03 14:30:27 [async_llm.py:448] decode_step_shm[shm_1]: write+signal=0.049ms, wait_output=4.802ms (core=4.681ms, asyncio_bounce=0.121ms), consume=0.007ms, total=4.858ms
INFO 03-03 14:30:27 [asy

(EngineCore_DP0 pid=761829) INFO 03-03 14:30:27 [shm_tensor_channel.py:364] SharedMemoryTensorChannel wait_input_ready stats: count=99, avg=0.657ms, min=0.229ms, max=3.694ms
